In [ ]:
from pathlib import Path
import json
import warnings
import sys
import pickle

import numpy as np
import pandas as pd
import h5py
from scipy import stats

from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_theme(style='whitegrid')
sns.set_theme(style='ticks')

In [ ]:
repo_dir = Path("../../..")

In [ ]:
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))
    
from analysis.curve_fitting.src.fitting_functions import LOSS_FUNCTIONS
from analysis.curve_fitting.src.utils import apply_filters, load_yaml, convert_loss_parameters, convert_loss_parameters_batch

from visualization.src.utils import set_ticks, save_figs, apply_hiearchical_aggregation, BENCHMARK_NAME_MAPPING, BENCHMARK_COLORS, METRICS_COMPACT, GENERIC_GROUPING_COLUMNS
from visualization.src.visualize import plot_reg, plot_reg_bivariate, plot_confidence_intervals

In [ ]:
artifacts_dir = repo_dir / "extract_n_eval" / "artifacts"
save_dir = repo_dir / "visualization" / "paper" / "supp" / "figures"

assert artifacts_dir.exists(), f"artifacts directory {artifacts_dir} does not exist!"

In [ ]:
results_file = artifacts_dir / "pretraining_results_with_metadata.csv"

df_all = pd.read_csv(results_file)

In [ ]:
df_all

In [ ]:
analysis_dir = repo_dir / 'analysis'
config_dir = analysis_dir / 'curve_fitting/configs/scaling_compute'
results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results-cka_c_test'
# results_dir = analysis_dir / 'curve_fitting/outputs/fitting_results_scaling_compute'

In [ ]:
architectures = [
    'efficientnet',
    'resnet',
    'convnext',
    'vit',
    'architecture_average',
]

benchmarks = [
    # 'bs_fz',
    'bs_mh',
    'tvsd',
    'things_fmri',
    'nsd',
    'things_eeg1',
    'things_eeg2',
    'things_meg',
    'benchmark_average',
    
]

METRIC = 'cka_c_test'

In [ ]:
all_configs = {}

for arch in architectures:
    for bench in benchmarks:
        yaml_config = config_dir / f'{arch}/{bench}.yaml'
        all_configs[f"{arch}-{bench}"] = load_yaml(yaml_config)

# yaml_config = config_dir / f'architecture_average/benchmark_average.yaml'
# all_configs[f"architecture_average-benchmark_average"] = load_yaml(yaml_config)

In [ ]:
L_fit_dict = {key: config['fitting_parameters']['loss_function'] for key, config in all_configs.items()}
L_viz_dict = {key: config['visualization']['loss_function'] for key, config in all_configs.items()}
x_scale_dict = {key: float(config['fitting_parameters']['X_scaler']) for key, config in all_configs.items()}

In [ ]:
all_df = {
    name: apply_filters(df_all, config.get('data_filters', {}))
    for name, config in all_configs.items()
}

In [ ]:
optimized_params_dict = {}
opt_params_boot_dict = {}

for cfg in all_configs.keys():
    try:
        with open(results_dir / f'scaling_compute-{cfg}' / 'results.pkl', 'rb') as f:
            results = pickle.load(f)

        L_fit = L_fit_dict[cfg]
        L_viz = L_viz_dict[cfg]
        optimized_params_dict[cfg] = convert_loss_parameters(results['optimized_parameters'], L_fit, L_viz)

        # Convert bootstrapped parameters
        opt_params_boot = results['optimized_parameters_bootstrapped']
        opt_params_boot_dict[cfg] = convert_loss_parameters_batch(
            params=opt_params_boot,
            src_loss=L_fit,
            dst_loss=L_viz
        )
    except Exception as e:
        print(f"Could not load results for config {cfg}: {e}")

In [ ]:
x_extend = 15
X_str = r'$$\tilde{D}$$'
linewidth = 3.0
alpha_scatter = 0.2
alpha_ci = 0.2
alpha_fit = 1.0
fig_multiplier = 0.75
fig_multiplier = 2.0
figsize = (10, 8)
figsize = (10, 7)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])
palette = sns.color_palette("tab10", n_colors=len(architectures))


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=figsize, dpi=300)

for i, bench in enumerate(benchmarks):
    ax = axes.flatten()[i]
    for j, arch in enumerate(architectures):
        cfg = f"{arch}-{bench}"
        
        if cfg not in optimized_params_dict:
            print(f"Skipping config {cfg} as no optimized parameters found.")
            continue

        df_plot = all_df[cfg]
        optimized_params = optimized_params_dict[cfg]
        opt_params_boot = opt_params_boot_dict[cfg]
        L = LOSS_FUNCTIONS[L_viz_dict[cfg]]
        x_scaler = x_scale_dict[cfg]
        X = df_plot.pretraining_total_flops_estimate.values / x_scaler


        color = palette[j]
        sns.scatterplot(data=df_plot, x='pretraining_total_flops_estimate', y=METRIC, ax=ax, color=color, alpha=alpha_scatter)

        plot_reg(X, optimized_params, L, ax, color=color, x_extend=x_extend*300, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth, legend=True, alpha=alpha_fit)
        plot_confidence_intervals(X, opt_params_boot, L, ax, color=color, x_extend=x_extend*300, x_scaler=x_scaler, alpha=alpha_ci, percentile=95.0, invert_y=True)

        data = df_all.copy()
        data = data[data.backbone_source == 'timm']
        if bench != 'benchmark_average':
            data = data[data.benchmark_name.isin(df_plot['benchmark_name'].unique())]
            

        data = apply_hiearchical_aggregation(
            df=data,
            groupby_cols=GENERIC_GROUPING_COLUMNS,
            agg_cols=METRICS_COMPACT,
            agg_func='mean'
        )
        
        sns.scatterplot(
            data=data,
            x='pretraining_total_flops_estimate',
            y=METRIC,
            hue='backbone_arch_family',
            markers={'timm': 'X'},
            ax=ax,
            # palette=colors,
            style='backbone_source'
        )




    ax.set_xscale('log')
    ax.set_xlabel('Pretraining Samples (D)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=16, fontweight='bold')
    ax.set_title(bench, fontsize=20, fontweight='bold')
    ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

    
    ax.legend().remove()



    ax.spines[['right', 'top']].set_visible(False)

plt.tight_layout()




# figures_dir = '../figures'
# fig_name = 'fig3_sample_scaling_full'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)

In [ ]:
x_extend = 15
X_str = r'$$\tilde{D}$$'
linewidth = 3.0
alpha_scatter = 0.1
alpha_ci = 0.1
alpha_fit = 1.0
alpha_fit = 0.7
fig_multiplier = 0.75
fig_multiplier = 0.6
# fig_multiplier = 2.0
# fig_multiplier = 1.0
figsize = (10, 8)
figsize = (11, 7)
figsize = (fig_multiplier * figsize[0], fig_multiplier * figsize[1])


In [ ]:
fig, axes = plt.subplots(1, 1, figsize=figsize, dpi=300)
axes = np.array([axes])

for i, arch in enumerate(['architecture_average']):  # enumerate(benchmarks):
    ax = axes.flatten()[i]
    for j, bench in enumerate(benchmarks):
        cfg = f"{arch}-{bench}"
        
        if cfg not in optimized_params_dict:
            print(f"Skipping config {cfg} as no optimized parameters found.")
            continue

        df_plot = all_df[cfg]
        optimized_params = optimized_params_dict[cfg]
        opt_params_boot = opt_params_boot_dict[cfg]
        L = LOSS_FUNCTIONS[L_viz_dict[cfg]]
        x_scaler = x_scale_dict[cfg]
        X = df_plot.pretraining_total_flops_estimate.values / x_scaler

        color = BENCHMARK_COLORS[bench]
        sns.scatterplot(data=df_plot, x='pretraining_total_flops_estimate', y=METRIC, ax=ax, color=color, alpha=alpha_scatter, label=bench)
        
        linewidth_ = linewidth
        alpha_fit_ = alpha_fit
        if bench == 'benchmark_average':
            linewidth_ = linewidth * 3.0
            alpha_fit_ = 1.0

        plot_reg(X, optimized_params, L, ax, color=color, x_extend=x_extend*300, linestyle='-', X_str=X_str, x_scaler=x_scaler, show_x_scaler=False, linewidth=linewidth_, legend=True, alpha=alpha_fit_)
        plot_confidence_intervals(X, opt_params_boot, L, ax, color=color, x_extend=x_extend*300, x_scaler=x_scaler, alpha=alpha_ci, percentile=95.0, invert_y=True)

        data = df_all.copy()
        data = data[data.backbone_source == 'timm']
        if bench != 'benchmark_average':
            data = data[data.benchmark_name.isin(df_plot['benchmark_name'].unique())]
        data = apply_hiearchical_aggregation(
            df=data,
            groupby_cols=GENERIC_GROUPING_COLUMNS,
            agg_cols=METRICS_COMPACT,
            agg_func='mean'
        )
        sns.scatterplot(
            data=data,
            x='pretraining_total_flops_estimate',
            y=METRIC,
            # hue='backbone_arch_family',
            color=color,
            # errorbar='sd',
            # errorbar=None,
            s=100,
            markers={
                'timm': 'X',
                'spvvs': 'o'
            },
            ax=ax,
            # palette=colors,
            style='backbone_source',
            # zorder=10
        )




    ax.set_xscale('log')
    ax.set_xlabel('Pretraining FLOPs (C)', fontsize=16, fontweight='bold')
    ax.set_ylabel('Alignment Score (S)', fontsize=16, fontweight='bold')
    # ax.set_title('Scaling of Alignment Score with Pretraining Compute', fontsize=20, fontweight='bold')
    # ax = set_ticks(ax, xticks_mode='log', yticks_mode=None, yticks=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

    handles, labels = ax.get_legend_handles_labels()
    labels_curves2 = [
        "spvvs",
        "timm",
    ]
    handles_curves2 = [handles[-3], handles[-1]]
    legend_curves2 = ax.legend(handles_curves2, labels_curves2, fontsize=12, loc='lower right', title='Model Source', title_fontsize=14)
    
    labels_curves = [
        BENCHMARK_NAME_MAPPING[benchmarks[j]] + '   ' + labels[j]
        for j in range(len(benchmarks))
    ]
    labels_curves = [
        BENCHMARK_NAME_MAPPING[benchmarks[j]]
        for j in range(len(benchmarks))
        
    ]
    handles_curves = handles[1::3]
    legend_curves = ax.legend(
        handles_curves, 
        labels_curves, 
        fontsize=12, 
        loc='upper right', 
        title='Benchmark',
        title_fontsize=14, 
        # bbox_to_anchor=(1.15, 1.0)
    )
    
    ax.add_artist(legend_curves2)
    
    # ax.legend().remove()



    ax.spines[['right', 'top']].set_visible(False)
    
# ax.legend().remove()

plt.tight_layout()




# figures_dir = save_dir
# fig_name = 'fig2_scaling_compute_summary'
# formats = ['pdf', 'png', 'svg']
# save_figs(figures_dir, fig_name, formats=formats)


figures_dir = save_dir
fig_name = f'metrics-pretraining-{METRIC}-compute-summary'
formats = ['pdf', 'png', 'svg']
save_figs(fig=fig, save_dir=figures_dir, base_filename=fig_name, dpi=300, formats=formats)